In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import einops as eo
import os
import rasterio as rio
from rasterio.features import rasterize
from rasterio.enums import Resampling
import matplotlib.pyplot as plt

from c2s.models.chicken import ChickenConfig, ChickenOutput, Chicken

In [ ]:
# Data pipeline parameters

# Data directory - should contain a subdirectory for each site
data_dir = '/network/scratch/m/matthew.fortier/lichen'

# File to put CSV data into
csv_file = 'dataset_large_vit.csv'

# which size chicken to use
vit_size = 'large'

# Classes can be commented out here if we wish to exclude them
class_map = {
    'lichen': 1,
    'rock': 4,
    'broadleaf': 5,
    'needleleaf': 6,
    'deadwood': 7,
    'graminoids': 8,
    'moss': 9,
    'soil': 10,
    'low_green': 12,
    'dry_branches': 13,
}

In [ ]:
sites = os.listdir(data_dir)
print(sites)

In [ ]:
def process_overlapping_shapes(label_gdf, chm_gdf):
    # Takes label and chm geometries, and returns a combined gdf
    # where each label is associated with a single canopy height
    combined_gdf = label_gdf.copy()
    combined_gdf['DN'] = -1
    chm_geometry = chm_gdf.geometry
    
    # iterate through each label shape
    for index, row in label_gdf.iterrows():
        intersecting_bools  = chm_geometry.intersects(row.geometry)
        intersecting_shapes = intersecting_bools[intersecting_bools != False]
        chm_values = list(chm_gdf.iloc[intersecting_shapes.index]['DN'])
        if len(chm_values) > 0:
            if row['class'] in [5, 6, 13]: # ['broadleaf', 'needleleaf', 'dry_branches']
                label_chm = max(chm_values)
            elif row['class'] in [1, 4, 7, 8, 9, 10, 12]: # ['lichen', 'rock', 'deadwood', 'graminoids', 'moss', 'soil', 'low_green']
                label_chm = min(chm_values)
            combined_gdf.loc[index, 'DN'] = label_chm
        else:
            combined_gdf.drop([index], inplace=True)
    return combined_gdf

In [ ]:
def load_site_data(site: str):
    # Retrieve class, RGB, canopy height, and certainty data from source files
    # Output them as a 6 layer raster (class, R, G, B, chm, certainty)
    
    rgb_file = os.path.join(data_dir, site, f'{site}_hp_transparent_mosaic_group1.tif')
    chm_file = os.path.join(data_dir, site, f'{site}_hp_chm_stratified.tif')
    chm_shp_file = os.path.join(data_dir, site, f'{site}_hp_chm_stratified.shp')
    label_file = os.path.join(data_dir, site, f'{site}_hp_labelledpoints.gpkg')

    # load RGB raster and metadata
    with rio.open(rgb_file) as f:
        # Get raster metadata
        height = f.height
        width = f.width
        crs = f.crs
        transform = f.transform
        output_raster = np.ndarray((6,height,width), dtype=np.uint8)
        rgb_raster = f.read((1,2,3), out=output_raster[1:4,:,:])
        
    # load shapefiles
    label_gdf = gpd.read_file(label_file).to_crs(crs).dropna(subset=['class', 'certainty'])
    label_gdf['certainty'] = label_gdf['certainty'].astype(np.uint8)
    chm_gdf = gpd.read_file(chm_shp_file).to_crs(crs)
    
    combined_gdf = process_overlapping_shapes(label_gdf, chm_gdf)
    
    rasterize(
        ((r['geometry'], r['class']) for _, r in combined_gdf.iterrows()),
        out_shape=(height, width),
        out=output_raster[0,:,:],
        transform=transform
    )
    
    rasterize(
        ((r['geometry'], r['DN']) for _, r in combined_gdf.iterrows()),
        out_shape=(height, width),
        out=output_raster[4,:,:],
        transform=transform
    )
    
    rasterize(
        ((r['geometry'], r['certainty']) for _, r in combined_gdf.iterrows()),
        out_shape=(height, width),
        out=output_raster[5,:,:],
        transform=transform
    )
    '''
    # labels are stored as shapefiles, so we will need to rasterize it
    label_gdf = gpd.read_file(label_file).to_crs(crs).dropna(subset=['class'])
    label_raster = rasterize(
        ((r['geometry'], r['class']) for _, r in label_gdf.iterrows()),
        out_shape=(height, width),
        transform=transform,
        dtype='uint8'
    )
    certainty_raster = rasterize(
        ((r['geometry'], r['certainty']) for _, r in label_gdf.iterrows()),
        out_shape=(height, width),
        transform=transform,
        dtype='uint8'
    )
    '''
    return output_raster

In [ ]:
def convert_to_dataframe(combined_raster, site):
    # Get a list of indices where we have labelled pixels
    # Input is the combined raster as above
    # Returns dataframe of class pixels along with indices [ [348, 1451], [581, 2099], ... ]
    indices = np.argwhere(np.isin(combined_raster[0], list(class_map.values())))
    
    labels_array = combined_raster[0, indices[:,0], indices[:,1]].reshape(-1, 1)
    labels_df = pd.DataFrame(labels_array, columns=['veg_class'])
    
    rgb_features = combined_raster[1:4, indices[:,0], indices[:,1]].T
    rgb_df = pd.DataFrame(rgb_features, columns=['R', 'G', 'B'])
    
    chm_features = combined_raster[4, indices[:,0], indices[:,1]].reshape(-1, 1)
    chm_df = pd.DataFrame(chm_features, columns=['chm'])
    
    certainty_array = combined_raster[5, indices[:,0], indices[:,1]].reshape(-1, 1)
    certainty_df = pd.DataFrame(certainty_array, columns=['class_certainty'])
    
    site_df = pd.DataFrame({'site': [site] * len(labels_df), 'y_pos': indices[:,0], 'x_pos': indices[:,1]})
    
    df = pd.concat([site_df, labels_df, rgb_df, chm_df, certainty_df], axis=1)
    return df, indices

In [ ]:
def get_chicken_feature_map(rgb_raster, vit_size="small"):
    """ Use sliding-window DinoV2 to extract visual features
    rgb_raster: torch tensor with shape (B, C, H, W)
    returns:    torch tensor with shape (B, num_features, H, W)
    """
    rgb_torch_raster = torch.from_numpy(rgb_raster.astype(np.float32)).unsqueeze(0)
    b, c, h, w = rgb_torch_raster.shape
    new_h = int(h/14)*14
    new_w = int(w/14)*14
    rgb_cropped = rgb_torch_raster[:,:,:new_h,:new_w]
    chicken_config = ChickenConfig(
        vit_size=vit_size,
        slider_buffer_size=1
    )
    chicken = Chicken(chicken_config)
    op = chicken._chicken_slider_infer(rgb_cropped).detach().numpy()
    return op

In [ ]:
def extract_chicken_pixel_features(chicken_feature_raster, indices):
    # Find the relevant feature vector for labeled pixels. Since texture featuremap is downsampled,
    # we need to modify the indices we use.
    chicken_indices = (indices/14).astype(int)
    chicken_pixel_features = chicken_feature_raster[0][:,chicken_indices[:,0],chicken_indices[:,1]].T
    chicken_df = pd.DataFrame(chicken_pixel_features, columns=[f'chicken_{i}' for i in range(chicken_pixel_features.shape[1])])
    return chicken_df

In [ ]:
all_features = None

In [ ]:
for site in sites:
    print(f'Processing {site}...')
    combined_raster = load_site_data(site)
    df1, indices = convert_to_dataframe(combined_raster, site)
    break
    # texture features
    print('  extracting texture features...')
    chicken_feature_map = get_chicken_feature_map(combined_raster[1:4], vit_size=vit_size)
    df2 = extract_chicken_pixel_features(chicken_feature_map, indices)

    df = pd.concat([df1,df2], axis=1)

    if all_features is None:
        all_features = df
    else:
        all_features = pd.concat([all_features, df], axis=0)
    

In [ ]:
all_features

In [ ]:
# Some additional cleanup
df = all_features
# Normalize colors
for c in ['R', 'G', 'B']:
    df[c] = df[c].astype(float)/255.0

In [ ]:
df

In [ ]:
df.to_csv(csv_file, index=False)